# 알약 검출 베이스라인 — Google Colab

**사전 준비:** 런타임 → 런타임 유형 변경 → **GPU (T4)** 선택

## 세션이 끊겼으면 셀 1번부터 / 세션 유지중이면 필요한 셀부터

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 설정
> `PROJECT_DIR` 만 각자 Drive 경로에 맞게 수정하세요.

In [ ]:
import os

PROJECT_DIR  = '/content/drive/MyDrive/베이스라인_코랩버전v1.2'  # ← 여기만 수정
BASELINE_DIR = os.path.join(PROJECT_DIR, 'baseline')
DATA_DIR     = os.path.join(PROJECT_DIR, 'sprint_ai_project1_data')

for p in [PROJECT_DIR, BASELINE_DIR, DATA_DIR]:
    print(f'{p} → exists={os.path.exists(p)}')

## 3. 작업 디렉토리 설정

In [ ]:
import os

os.chdir(BASELINE_DIR)
print('현재 디렉토리:', os.getcwd())
print('내용:', os.listdir('.'))

## 4. config 경로 업데이트

In [ ]:
import yaml

cfg_path = 'configs/default.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

cfg['data']['data_root'] = DATA_DIR
cfg['data']['processed_dir'] = './data/processed'
cfg['data']['num_workers'] = 2
cfg['train']['epochs'] = 10  # 테스트는 1, 본학습은 30

with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print('data_root:', cfg['data']['data_root'])
print('epochs   :', cfg['train']['epochs'])

## 5. 패키지 설치

In [ ]:
!pip install -q -r requirements.txt

## 6. GPU 확인

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 7. 어노테이션 전처리 (최초 1회)

In [ ]:
!python scripts/preprocess.py

## 8. (선택) GT 박스 시각화

In [ ]:
!python scripts/visualize.py --n 3 --output outputs/vis

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

vis_files = sorted(glob.glob('outputs/vis/*.jpg'))[:3]
fig, axes = plt.subplots(1, len(vis_files), figsize=(15, 6))
for ax, path in zip(axes, vis_files):
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    ax.set_title(os.path.basename(path), fontsize=8)
plt.tight_layout()
plt.show()

---
## [Faster R-CNN] 학습 & 추론

In [ ]:
# Faster R-CNN 학습
!python train.py

In [ ]:
# Faster R-CNN 추론 → outputs/predictions/submission.csv
!python inference.py --checkpoint outputs/checkpoints/best.pth

---
## [YOLO11] 학습 & 추론
> Faster R-CNN 대신 사용. 셀 순서: 패키지설치 → 데이터변환 → 학습 → 추론

In [ ]:
# YOLO 패키지 설치
!pip install -q ultralytics

In [ ]:
# YOLO 데이터 변환 (최초 1회)
!python scripts/convert_to_yolo.py

In [ ]:
# YOLO 학습
# 모델 크기: yolo11n(빠름) / yolo11s / yolo11m(추천) / yolo11l / yolo11x(최고성능)
!python train_yolo.py --model yolo11m.pt --epochs 30 --batch 8

In [ ]:
# YOLO 추론 → outputs/predictions/submission_yolo.csv
!python inference_yolo.py --checkpoint outputs/yolo/train/weights/best.pt

---
## 틀린 이미지 시각화

In [ ]:
# 한글 폰트 설치 (최초 1회)
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
print('폰트 설치 완료')

In [ ]:
!python scripts/visualize_errors.py --n 10 --output outputs/errors

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

error_files = sorted(glob.glob('outputs/errors/*.jpg'))
print(f'틀린 이미지 {len(error_files)}개')

for path in error_files[:10]:
    fig, ax = plt.subplots(1, figsize=(6, 8))
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    ax.set_title(os.path.basename(path), fontsize=9)
    plt.tight_layout()
    plt.show()

---
## Drive 백업

In [ ]:
import shutil, os

BACKUP_DIR = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(BACKUP_DIR, exist_ok=True)

for folder in ['checkpoints', 'predictions', 'yolo']:
    src = os.path.join('outputs', folder)
    if os.path.exists(src):
        shutil.copytree(src, os.path.join(BACKUP_DIR, folder), dirs_exist_ok=True)
        print(f'{folder} 백업 완료')

print('Drive 백업 완료:', BACKUP_DIR)